## Demostrate Static Embeddings and Sementic Search
#### GloVe - pre-trained word embedding model
- The pretrained model has already learned that words appearing in similar contexts tend to have similar vectors.
- Models
-       glove-wiki-gigaword-50 → small, fast, easy
-       glove-wiki-gigaword-100 → slightly better quality
-       word2vec-google-news-300 → better quality, much larger

In [1]:
import gensim.downloader as api
import numpy as np

# -------------------------------------------------
# 1. Load a pretrained STATIC word embedding model
# -------------------------------------------------
# This downloads the model the first time you run it.
# It is relatively small and good for demos.
model = api.load("glove-wiki-gigaword-50")

# -------------------------------------------------
# 2. Tokenize a word (very simple tokenization)
# -------------------------------------------------
def tokenize_word(word: str) -> str:
    return word.lower().strip()

# -------------------------------------------------
# 3. Get embedding vector for a token
# -------------------------------------------------
def get_embedding(token: str):
    if token in model:
        return model[token]
    else:
        return None
# -------------------------------------------------

## Cosine Similarity
Cosine similarity measures how similar two vectors are by computing the cosine of the angle between them, focusing on their direction rather than their magnitude


$$
\text{cosine\_similarity}(v_{target}, v_{input}) = \cos(\theta) = 
\frac{v_{target} \cdot v_{input}}
{\|v_{target}\| \, \|v_{input}\|}
$$

#### Where:
1. Dot product (numerator)
$$
v_{target} \cdot v_{input} = 
\sum_{i=1}^{d} (v_{target,i} \times v_{input,i})
$$
2. Magnitudes (denominator)
$$
\|v_{target}\| = \sqrt{\sum_{i=1}^{d} v_{target,i}^2}
$$
$$
\|v_{input}\| = \sqrt{\sum_{i=1}^{d} v_{input,i}^2}
$$

In [2]:
# -------------------------------------------------
# 4. Semantic (Cosine Similarity) search across the model vocabulary
# -------------------------------------------------
def semantic_search(token: str, topn: int = 10):
    if token not in model:
        return []
    # The most_similar method returns a list of (word, similarity) tuples
    # This is a Cosine similarity search for a given token against the entire vocabulary
    # Measures angle between vectors, not their magnitude
    return model.most_similar(token, topn=topn)
# -------------------------------------------------

## Embedding Demo

In [3]:
# -------------------------------------------------
# 6. Demo
# -------------------------------------------------
query_word = "bank"

topn_n = 10
token = tokenize_word(query_word)

print("-"*68)
print("--- Word Embedding Demo ---")
print("-"*68)
print(f"Model Used: GloVe (Wikipedia + Gigaword, 50 dimensions)")
print(f"No. of Dimensions: {model.vector_size}")
print(f"Vocabulary Size (No. of Tokens): {len(model.key_to_index)}")
print(f"That means this model has {model.vector_size} x {len(model.key_to_index)} = {model.vector_size * len(model.key_to_index) / 10**6} million parameters in total")
print("-"*68 + "\n")


embedding = get_embedding(token)

if embedding is None:
    print(f"Token '{token}' not found in vocabulary.")
else:
    print(f"Original word : {query_word}")
    print(f"Token         : {token}")
    print(f"Token Index   : {model.key_to_index.get(token, 'Not found in vocabulary')}")

    print("First 3 (dimensions) values of embedding :")
    print(embedding[:3])

    print(f"\nThe vector for '{token}' (all {embedding.shape[0]} dimensions  of embedding) :")
    print("-- A vector is a list of numbers that represents an object in a n-dimensional space.")
    print(embedding)

    print("\n" + "-"*68)
    print(f"Top {topn_n} related words from full vocabulary:")
    print("-"*68)
    for word, score in semantic_search(token, topn=topn_n):
        print(f"{word:15s}  similarity={score:.4f}")
    print("-"*68)


--------------------------------------------------------------------
--- Word Embedding Demo ---
--------------------------------------------------------------------
Model Used: GloVe (Wikipedia + Gigaword, 50 dimensions)
No. of Dimensions: 50
Vocabulary Size (No. of Tokens): 400000
That means this model has 50 x 400000 = 20.0 million parameters in total
--------------------------------------------------------------------

Original word : bank
Token         : bank
Token Index   : 231
First 3 (dimensions) values of embedding :
[ 0.66488 -0.11391  0.67844]

The vector for 'bank' (all 50 dimensions  of embedding) :
-- A vector is a list of numbers that represents an object in a n-dimensional space.
[ 0.66488  -0.11391   0.67844   0.17951   0.6828   -0.47787  -0.30761
  0.17489  -0.70512  -0.55022   0.1514    0.10214  -0.45063  -0.33069
  0.056133  1.2271    0.55607  -0.68297   0.037364  0.70266   1.9093
 -0.61483  -0.83329  -0.3023   -1.1118   -1.55      0.2604    0.22957
 -1.0375   -0.31

## Lets Plot the Vectors in 3D
- We are using PCA (Principal Component Analysis) which is a mathematical technique that compresses high‑dimensional data into fewer dimensions while preserving as much important information as possible.
- Note: Since we are plotting 50 dimensions on to 3 dimensional plot some distortion of the geometry is expected

In [4]:
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go

def plot_related_words_3d(query_word, topn=6):
    token = tokenize_word(query_word)
    similar_words = semantic_search(token, topn=topn)

    if not similar_words:
        print(f"No similar words found for '{query_word}'")
        return

    words = [token] + [word for word, score in similar_words]
    vectors = np.array([model[word] for word in words])

    pca = PCA(n_components=3)
    coords = pca.fit_transform(vectors)

    query_coord = coords[0]
    related_coords = coords[1:]

    related_words = [word for word, score in similar_words]
    related_scores = [score for word, score in similar_words]

    fig = go.Figure()

    # Related words
    fig.add_trace(go.Scatter3d(
        x=related_coords[:, 0],
        y=related_coords[:, 1],
        z=related_coords[:, 2],
        mode="markers+text",
        text=related_words,
        textposition="top center",
        marker=dict(size=6, color="steelblue"),
        customdata=np.array(related_scores).reshape(-1, 1),
        hovertemplate=(
            "<b>%{text}</b><br>" +
            "Similarity: %{customdata[0]:.4f}<br>" +
            "PCA1: %{x:.3f}<br>" +
            "PCA2: %{y:.3f}<br>" +
            "PCA3: %{z:.3f}<extra></extra>"
        ),
        name="Related words"
    ))

    # Query word
    fig.add_trace(go.Scatter3d(
        x=[query_coord[0]],
        y=[query_coord[1]],
        z=[query_coord[2]],
        mode="markers+text",
        text=[token],
        textposition="top center",
        marker=dict(size=10, color="red"),
        hovertemplate=(
            "<b>%{text}</b><br>" +
            "Query word<br>" +
            "PCA1: %{x:.3f}<br>" +
            "PCA2: %{y:.3f}<br>" +
            "PCA3: %{z:.3f}<extra></extra>"
        ),
        name="Query word"
    ))

    fig.update_layout(
        title=f"3D Static Embedding Plot for '{query_word}'",
        scene=dict(
            xaxis_title="PCA Dimension 1",
            yaxis_title="PCA Dimension 2",
            zaxis_title="PCA Dimension 3"
        ),
        width=1000,
        height=750
    )

    fig.show()

    print(f"\nQuery word: {query_word}")
    print("Top related words:")
    for word, score in similar_words:
        print(f"{word:12s} similarity = {score:.4f}")

In [5]:
# --------------------------------------------------
# 6. Plot 3D
# --------------------------------------------------
plot_related_words_3d("father", topn=6)


Query word: father
Top related words:
son          similarity = 0.9529
brother      similarity = 0.9323
grandfather  similarity = 0.9146
friend       similarity = 0.9048
uncle        similarity = 0.8977
mother       similarity = 0.8909


## Vector Arithmatic - The Power of Vectors
Vector addition and subtraction create a new “target” direction by combining or offsetting meanings, and cosine similarity then measures how closely other vectors align with this resulting direction in embedding space.


In [6]:
# ---------------------------------------------------------
# 1. Family words to visualize
# ---------------------------------------------------------
family_words = ["man", "woman", "father", "mother", "uncle", "aunt"]

# Keep only words that exist in the model vocabulary
family_words = [w for w in family_words if w in model.key_to_index]
print("Words found in vocab:", family_words)

# ---------------------------------------------------------
# 2. Get vectors and reduce to 3D
# ---------------------------------------------------------
vectors = np.array([model[w] for w in family_words])

pca = PCA(n_components=3)
coords = pca.fit_transform(vectors)

# ---------------------------------------------------------
# 3. Colors and sizes
# ---------------------------------------------------------
colors = []
sizes = []

for word in family_words:
    if word == "mother":
        colors.append("crimson")
        sizes.append(10)
    elif word in ["woman", "aunt"]:
        colors.append("deeppink")
        sizes.append(8)
    elif word in ["man", "father", "uncle"]:
        colors.append("steelblue")
        sizes.append(8)
    else:
        colors.append("gray")
        sizes.append(6)

# ---------------------------------------------------------
# 4. Interactive Plotly 3D plot
# ---------------------------------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=coords[:, 0],
    y=coords[:, 1],
    z=coords[:, 2],
    mode="markers+text",
    text=family_words,
    textposition="top center",
    marker=dict(size=sizes, color=colors),
    name="Family words"
))

# Optional helper lines
pair_lines = [("man", "woman"), ("father", "mother"), ("uncle", "aunt")]
word_to_coord = {w: coords[i] for i, w in enumerate(family_words)}

for a, b in pair_lines:
    if a in word_to_coord and b in word_to_coord:
        pa = word_to_coord[a]
        pb = word_to_coord[b]
        fig.add_trace(go.Scatter3d(
            x=[pa[0], pb[0]],
            y=[pa[1], pb[1]],
            z=[pa[2], pb[2]],
            mode="lines",
            line=dict(color="gray", width=4),
            showlegend=False
        ))

fig.update_layout(
    title="3D Plot of Family Word Embeddings",
    scene=dict(
        xaxis_title="PCA 1",
        yaxis_title="PCA 2",
        zaxis_title="PCA 3"
    ),
    width=1000,
    height=750
)

fig.show()

Words found in vocab: ['man', 'woman', 'father', 'mother', 'uncle', 'aunt']


#### Lets do the arithmatic

- Part 1: Using Query Vector

In [7]:
# ---------------------------------------------------------
# 1. Cosine similarity function
# ---------------------------------------------------------
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# ---------------------------------------------------------
# 2. Analogy query
# ---------------------------------------------------------
query_vec = model["father"] - model["man"] + model["woman"]

print("Analogy query: father - man + woman")
#print("\nQuery vector shape:", query_vec.shape)

# ---------------------------------------------------------
# 3. Rank only the family words
# ---------------------------------------------------------
candidate_scores = []

for word in family_words:
    score = cosine_similarity(model[word], query_vec)
    candidate_scores.append((word, score))

candidate_scores.sort(key=lambda x: x[1], reverse=True)

print("\nNearest among selected family words:")
for word, score in candidate_scores:
    print(f"{word:10s} cosine={score:.4f}")


Analogy query: father - man + woman

Nearest among selected family words:
mother     cosine=0.9371
father     cosine=0.9009
woman      cosine=0.8188
aunt       cosine=0.8025
uncle      cosine=0.7495
man        cosine=0.6399


- Part 2: Nearest Neighbors in Full Vocabulary 

In [8]:
# ---------------------------------------------------------
# 4. Ask gensim for full-vocabulary nearest neighbors
# ---------------------------------------------------------
print("\nTop full-vocabulary analogy results:")
full_results = model.most_similar(
    positive=["father", "woman"],
    negative=["man"],
    topn=10
)

for word, score in full_results:
    print(f"{word:15s} cosine={score:.4f}")



Top full-vocabulary analogy results:
mother          cosine=0.9297
daughter        cosine=0.9290
wife            cosine=0.9038
grandmother     cosine=0.8817
husband         cosine=0.8662
married         cosine=0.8473
niece           cosine=0.8470
granddaughter   cosine=0.8349
widow           cosine=0.8248
son             cosine=0.8240
